In [1]:
import paramiko
import pandas as pd
from io import StringIO
import sqlite3
import tempfile
import os

def read_db_directly():
    config = {'host': '80.225.228.224','username': 'ubuntu','private_key': r'D:\NIFTY_Options_21ema_strategy\21ema_strategy_v2_Current_Working_Dec2025\oracle_key\ssh-key-2025-12-20.key',}
    
    # remote_db_path = '/home/ubuntu/21EMA_Trading_Logs.db'
    remote_db_path = '/home/ubuntu/final_trading_logs.db'    
    
    try:
        print(f"Connecting to {config['host']}...")
        private_key = paramiko.RSAKey.from_private_key_file(config['private_key'])
        
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=config['host'],username=config['username'],pkey=private_key)
        
        print("Connected! Reading database...")
        
        # Create a temporary file to store the database
        with tempfile.NamedTemporaryFile(suffix='.db', delete=False) as temp_file:
            temp_path = temp_file.name
        
        # Download the database file
        sftp = ssh.open_sftp()
        sftp.get(remote_db_path, temp_path)
        sftp.close()
        
        # Read from the temporary database file
        conn = sqlite3.connect(temp_path)
        # signal_df = pd.read_sql_query("SELECT * FROM ema_5min_logs", conn)
        signal_df = pd.read_sql_query("SELECT * FROM trading_logs", conn)


        conn.close()
        
        # Clean up temporary file
        os.unlink(temp_path)
        
        # Display the data
        # print(f"\nTotal records: {len(signal_df)}")
        # display_cols = ['instrument', 'strikesymbol', 'entry_time', 'entry_price',
        #                'stoploss', 'target', 'pos_size', 'exit_time',
        #                'exit_reason', 'exit_price', 'option_profit']
        
        # existing_cols = [col for col in display_cols if col in signal_df.columns]
        # print(signal_df[existing_cols].tail(20))
        
        ssh.close()
        return signal_df
        
    except Exception as e:
        print(f"Error: {e}")
        return None

In [2]:
df = read_db_directly()

Connecting to 80.225.228.224...
Connected! Reading database...
